Студент: Гапуніч Анатолій

Група: AI&ML 4

# Фінальний проєкт GoIT RDB — Infectious Cases

**Середовище:** Google Colab · PostgreSQL 16 · Psycopg 3 · pandas  
**Схема:** `pandemic`  
**Джерело:** `infectious_cases.csv`, наданий у завданні.

У цьому notebook послідовно виконуються всі п’ять завдань: імпорт CSV, нормалізація до 3НФ, агрегація `Number_rabies`, побудова різниці в роках вбудованими SQL-функціями та створення двох власних SQL-функцій.

> Запускайте через **Runtime → Restart session and run all**. Встановлення PostgreSQL 16 і підключення зроблено так само, як у попередніх домашніх роботах: офіційний PGDG APT-репозиторій, `pg_ctlcluster`, роль `root` і Psycopg 3 через Unix socket. `pgserver` не використовується.

## 0. Підготовка середовища PostgreSQL 16

Встановлюємо сервер і клієнт PostgreSQL саме 16-ї версії з офіційного репозиторію PGDG. Після інсталяції запускаємо локальний кластер та створюємо роль PostgreSQL `root`, яка відповідає системному користувачу Colab. Це дає змогу підключатися через локальну peer-аутентифікацію без пароля.

In [1]:
%%bash
set -euo pipefail

# Install prerequisites.
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ca-certificates curl

# Add the official PostgreSQL APT repository and signing key.
install -d /usr/share/postgresql-common/pgdg
curl -fsSL https://www.postgresql.org/media/keys/ACCC4CF8.asc \
  -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc

CODENAME="$(. /etc/os-release && printf '%s' "${VERSION_CODENAME:-}")"
ARCH="$(dpkg --print-architecture)"
if [[ -z "${CODENAME}" ]]; then
  echo "Не вдалося визначити VERSION_CODENAME із /etc/os-release." >&2
  exit 1
fi

cat > /etc/apt/sources.list.d/pgdg.sources <<EOF
Types: deb
URIs: https://apt.postgresql.org/pub/repos/apt
Suites: ${CODENAME}-pgdg
Architectures: ${ARCH}
Components: main
Signed-By: /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc
EOF

# Install PostgreSQL 16 server and client.
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql-16 postgresql-client-16

# Start the default PostgreSQL 16 cluster.
pg_ctlcluster 16 main start || pg_ctlcluster 16 main status

# Create a PostgreSQL superuser matching Colab's OS user for local peer auth.
if ! runuser -u postgres -- psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='root'" | grep -q '^1$'; then
  runuser -u postgres -- createuser --superuser root
fi

# Install the same Python database stack used in the previous assignments.
python -m pip install -q "psycopg[binary]" pandas matplotlib

# Print the actual server version.
runuser -u postgres -- psql -tAc "SHOW server_version;"

Preconfiguring packages ...
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../ca-certificates_20260816ubuntu0.24.04.2_all.deb ...
Unpacking ca-certificates (20260816ubuntu0.24.04.2) over (20260816ubuntu0.24.04.1) ...
Setting up ca-certificates (20260816ubuntu0.24.04.2) ...
Updating certificates in /etc/ssl/certs...
0 added, 0 removed; done.
Running hooks in /etc/ca-certificates/update.d...
done.
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.015s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.015s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
done.
Preconfiguring packages ...
Selecting previously unselected package libjson-perl.
(Reading database ... 122

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 0.1. Підключення та допоміжні функції

Підключаємося до бази `postgres` через Unix socket. `execute_script()` виконує багатокомандні DDL/SQL-скрипти через `prepare=False`; вручну ділити SQL через `split(';')` не потрібно. `query_df()` повертає результат `SELECT` як pandas DataFrame. `timed_step()` додає часові мітки, щоб було видно, на якому етапі виникла помилка.

In [2]:
from pathlib import Path
from time import perf_counter
import zipfile

import pandas as pd
import psycopg
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

conn = psycopg.connect(
    dbname="postgres",
    user="root",
    host="/var/run/postgresql",
    connect_timeout=10,
)
conn.autocommit = False


def timed_step(name: str, operation):
    """Run one notebook operation and report elapsed time and status."""
    started_at = perf_counter()
    print(f"[START] {name}", flush=True)
    try:
        result = operation()
    except Exception as exc:
        conn.rollback()
        elapsed = perf_counter() - started_at
        print(
            f"[FAILED] {name} after {elapsed:.2f} sec: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )
        raise
    elapsed = perf_counter() - started_at
    print(f"[DONE] {name}: {elapsed:.2f} sec", flush=True)
    return result


def execute_script(sql_script: str) -> None:
    """Execute a multi-statement SQL script in one transaction."""
    if conn.closed:
        raise RuntimeError("PostgreSQL connection is closed. Re-run the connection cell.")
    try:
        with conn.cursor() as cur:
            # Psycopg simple-query mode supports multiple SQL statements.
            # Do not split SQL by semicolons: they can occur in comments/literals.
            cur.execute(sql_script, prepare=False)
        conn.commit()
    except Exception:
        conn.rollback()
        raise


def query_df(sql_query: str) -> pd.DataFrame:
    """Execute a row-returning SQL query and return a pandas DataFrame."""
    if conn.closed:
        raise RuntimeError("PostgreSQL connection is closed. Re-run the connection cell.")
    try:
        with conn.cursor() as cur:
            cur.execute(sql_query, prepare=False)
            if cur.description is None:
                raise ValueError("query_df() expects a SELECT query returning rows.")
            rows = cur.fetchall()
            columns = [column.name for column in cur.description]
        conn.commit()
        return pd.DataFrame(rows, columns=columns)
    except Exception:
        conn.rollback()
        raise


def save_query_screenshot(title: str, sql_text: str, result: pd.DataFrame, filename: str, max_rows: int = 10):
    """Save a compact evidence image containing the SQL and its real DB result."""
    preview = result.head(max_rows).copy()

    def display_value(value):
        if pd.isna(value):
            return "NULL"
        return str(value)

    cell_text = [[display_value(value) for value in row] for row in preview.itertuples(index=False, name=None)]
    sql_lines = sql_text.strip().splitlines()
    if len(sql_lines) > 13:
        sql_display = "\n".join(sql_lines[:13]) + "\n-- ..."
    else:
        sql_display = "\n".join(sql_lines)

    fig, ax = plt.subplots(figsize=(17, 8))
    ax.axis("off")
    ax.text(0.01, 0.98, title, transform=ax.transAxes, fontsize=15,
            fontweight="bold", va="top")
    ax.text(0.01, 0.90, "SQL query", transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top")
    ax.text(0.01, 0.86, sql_display, transform=ax.transAxes,
            fontsize=7.2, family="monospace", va="top")

    table = ax.table(
        cellText=cell_text,
        colLabels=list(preview.columns),
        cellLoc="center",
        loc="lower center",
        bbox=[0.01, 0.02, 0.98, 0.43],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)

    output_path = SCREENSHOT_DIR / filename
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Screenshot saved: {output_path}")


# Keep all evidence files in predictable folders in the Colab working directory.
SCREENSHOT_DIR = Path.cwd() / "screenshots"
RESULT_DIR = Path.cwd() / "results"
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

version_df = query_df("SELECT current_setting('server_version') AS postgresql_version;")
display(version_df)
assert version_df.loc[0, "postgresql_version"].startswith("16."), "Потрібен PostgreSQL 16."

,postgresql_version
0,16.15 (Ubuntu 16.15-1.pgdg24.04+2)


## 1. Пошук і перевірка CSV

Файл може бути доступний у папці `data/` репозиторію або безпосередньо в `/content`. Якщо notebook не знаходить його, Colab запропонує завантажити `infectious_cases.csv`. Перед імпортом перевіряємо заголовок і порядок усіх 12 колонок: `COPY` читає значення позиційно.

In [3]:
EXPECTED_COLUMNS = [
    "Entity",
    "Code",
    "Year",
    "Number_yaws",
    "polio_cases",
    "cases_guinea_worm",
    "Number_rabies",
    "Number_malaria",
    "Number_hiv",
    "Number_tuberculosis",
    "Number_smallpox",
    "Number_cholera_cases",
]


def locate_csv() -> Path:
    """Find the supplied CSV or request it from the user in Google Colab."""
    candidates = [
        Path("/content/data/infectious_cases.csv"),
        Path("/content/infectious_cases.csv"),
        Path.cwd() / "data" / "infectious_cases.csv",
        Path.cwd() / "infectious_cases.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Не знайдено infectious_cases.csv. Покладіть файл у data/ або завантажте його в середовище."
        ) from exc

    print("Завантажте файл infectious_cases.csv з комп’ютера.")
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    selected_name = next(
        (name for name in uploaded_names if "infectious_cases" in name.lower()),
        next((name for name in uploaded_names if name.lower().endswith(".csv")), None),
    )
    if selected_name is None:
        raise FileNotFoundError("Не отримано CSV-файл.")

    target = Path("/content") / "infectious_cases.csv"
    target.write_bytes(uploaded[selected_name])
    return target


DATA_PATH = locate_csv()
source_df = pd.read_csv(DATA_PATH)

if list(source_df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        "Заголовки або порядок CSV-колонок відрізняються від очікуваних.\n"
        f"Очікується: {EXPECTED_COLUMNS}\n"
        f"Отримано:   {list(source_df.columns)}"
    )

EXPECTED_RAW_ROWS = len(source_df)
EXPECTED_ENTITY_PAIRS = source_df[["Entity", "Code"]].drop_duplicates().shape[0]
EXPECTED_RABIES_VALUES = int(source_df["Number_rabies"].notna().sum())

print(f"CSV: {DATA_PATH}")
print(f"Rows: {EXPECTED_RAW_ROWS:,}")
print(f"Unique Entity + Code pairs: {EXPECTED_ENTITY_PAIRS:,}")
print(f"Non-NULL Number_rabies: {EXPECTED_RABIES_VALUES:,}")
display(source_df.head())

Завантажте файл infectious_cases.csv з комп’ютера.


Saving infectious_cases.csv to infectious_cases.csv
CSV: /content/infectious_cases.csv
Rows: 10,521
Unique Entity + Code pairs: 245
Non-NULL Number_rabies: 6,780


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## 1.1. Створення схеми й сирої таблиці та імпорт CSV

Створюємо схему `pandemic` і таблицю `infectious_cases`. Колонки показників мають тип `NUMERIC`, а порожні значення CSV імпортуються як SQL `NULL`. Для завантаження використовуємо PostgreSQL `COPY FROM STDIN` через Psycopg 3 — це не потребує доступу серверного процесу до локального шляху файла.

In [4]:
DDL_RAW_SQL = r"""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic, public;

DROP VIEW IF EXISTS pandemic.v_infectious_cases_year_difference;
DROP TABLE IF EXISTS pandemic.infectious_indicators CASCADE;
DROP TABLE IF EXISTS pandemic.entities CASCADE;
DROP TABLE IF EXISTS pandemic.infectious_cases CASCADE;

CREATE TABLE pandemic.infectious_cases (
    entity                  TEXT NOT NULL,
    code                    TEXT,
    year                    INTEGER NOT NULL CHECK (year BETWEEN 1 AND 9999),
    number_yaws             NUMERIC,
    polio_cases             NUMERIC,
    cases_guinea_worm       NUMERIC,
    number_rabies           NUMERIC,
    number_malaria          NUMERIC,
    number_hiv              NUMERIC,
    number_tuberculosis     NUMERIC,
    number_smallpox         NUMERIC,
    number_cholera_cases    NUMERIC
);
"""

timed_step("Create schema and raw table", lambda: execute_script(DDL_RAW_SQL))

COPY_SQL = r"""
COPY pandemic.infectious_cases (
    entity, code, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
FROM STDIN WITH (FORMAT CSV, HEADER TRUE, NULL '');
"""


def copy_csv_to_postgres(path: Path) -> None:
    """Stream the CSV file to PostgreSQL using COPY FROM STDIN."""
    try:
        with conn.cursor() as cur:
            with cur.copy(COPY_SQL) as copy:
                with path.open("rb") as csv_file:
                    while True:
                        chunk = csv_file.read(1024 * 1024)
                        if not chunk:
                            break
                        copy.write(chunk)
        conn.commit()
    except Exception:
        conn.rollback()
        raise


timed_step("COPY CSV into pandemic.infectious_cases", lambda: copy_csv_to_postgres(DATA_PATH))

RAW_CHECK_SQL = r"""
SELECT
    COUNT(*) AS raw_rows,
    COUNT(*) FILTER (WHERE code IS NULL) AS rows_without_code,
    COUNT(*) FILTER (WHERE number_rabies IS NOT NULL) AS numeric_rabies_rows,
    COUNT(*) FILTER (WHERE number_rabies IS NULL) AS null_rabies_rows
FROM pandemic.infectious_cases;
"""
raw_check_df = query_df(RAW_CHECK_SQL)
display(raw_check_df)

assert int(raw_check_df.loc[0, "raw_rows"]) == EXPECTED_RAW_ROWS
assert int(raw_check_df.loc[0, "numeric_rabies_rows"]) == EXPECTED_RABIES_VALUES

RAW_PREVIEW_SQL = r"""
SELECT entity, code, year, number_rabies
FROM pandemic.infectious_cases
ORDER BY entity, year
LIMIT 10;
"""
raw_preview_df = query_df(RAW_PREVIEW_SQL)
display(raw_preview_df)
save_query_screenshot(
    "01 — Raw table pandemic.infectious_cases",
    RAW_PREVIEW_SQL,
    raw_preview_df,
    "01_raw_table_preview.png",
)
raw_preview_df.to_csv(RESULT_DIR / "raw_table_preview.csv", index=False)

[START] Create schema and raw table
[DONE] Create schema and raw table: 0.03 sec
[START] COPY CSV into pandemic.infectious_cases
[DONE] COPY CSV into pandemic.infectious_cases: 0.03 sec


,raw_rows,rows_without_code,numeric_rabies_rows,null_rabies_rows
0,10521,1128,6780,3741


,entity,code,year,number_rabies
0,Afghanistan,AFG,1960,None
1,Afghanistan,AFG,1965,None
2,Afghanistan,AFG,1980,None
3,Afghanistan,AFG,1981,None
4,Afghanistan,AFG,1982,None
5,Afghanistan,AFG,1983,None
6,Afghanistan,AFG,1984,None
7,Afghanistan,AFG,1985,None
8,Afghanistan,AFG,1986,None
9,Afghanistan,AFG,1987,None


Screenshot saved: /content/screenshots/01_raw_table_preview.png


## 2. Нормалізація до третьої нормальної форми

У таблиці `entities` зберігаємо один запис на унікальну пару `Entity` + `Code`. У `infectious_indicators` залишаємо рік і всі показники, а замість повторюваних текстових атрибутів використовуємо `entity_id`.

Для комбінацій із відсутнім `Code` важливо зберегти унікальність. PostgreSQL 16 підтримує `UNIQUE NULLS NOT DISTINCT`, тому `(entity, NULL)` поводиться як одна унікальна комбінація. Таблиця фактів має складений первинний ключ `(entity_id, year)`; у вихідних даних дубльованих пар сутність/рік немає.

In [5]:
NORMALIZATION_SQL = r"""
CREATE TABLE pandemic.entities (
    entity_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity      TEXT NOT NULL,
    code        TEXT,
    CONSTRAINT uq_entities_entity_code UNIQUE NULLS NOT DISTINCT (entity, code)
);

CREATE TABLE pandemic.infectious_indicators (
    entity_id               BIGINT NOT NULL REFERENCES pandemic.entities(entity_id),
    year                    INTEGER NOT NULL CHECK (year BETWEEN 1 AND 9999),
    number_yaws             NUMERIC,
    polio_cases             NUMERIC,
    cases_guinea_worm       NUMERIC,
    number_rabies           NUMERIC,
    number_malaria          NUMERIC,
    number_hiv              NUMERIC,
    number_tuberculosis     NUMERIC,
    number_smallpox         NUMERIC,
    number_cholera_cases    NUMERIC,
    CONSTRAINT pk_infectious_indicators PRIMARY KEY (entity_id, year)
);

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT BTRIM(entity), NULLIF(BTRIM(code), '')
FROM pandemic.infectious_cases;

INSERT INTO pandemic.infectious_indicators (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    e.entity_id,
    r.year,
    r.number_yaws,
    r.polio_cases,
    r.cases_guinea_worm,
    r.number_rabies,
    r.number_malaria,
    r.number_hiv,
    r.number_tuberculosis,
    r.number_smallpox,
    r.number_cholera_cases
FROM pandemic.infectious_cases AS r
JOIN pandemic.entities AS e
  ON e.entity = BTRIM(r.entity)
 AND e.code IS NOT DISTINCT FROM NULLIF(BTRIM(r.code), '');

CREATE INDEX idx_infectious_indicators_year
    ON pandemic.infectious_indicators(year);

ANALYZE pandemic.entities;
ANALYZE pandemic.infectious_indicators;
"""

timed_step("Normalize raw data into 3NF tables", lambda: execute_script(NORMALIZATION_SQL))

TABLE_SUMMARY_SQL = r"""
SELECT 'infectious_cases' AS table_name,
       COUNT(*) AS row_count,
       (SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = 'pandemic' AND table_name = 'infectious_cases') AS column_count
FROM pandemic.infectious_cases
UNION ALL
SELECT 'entities', COUNT(*),
       (SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = 'pandemic' AND table_name = 'entities')
FROM pandemic.entities
UNION ALL
SELECT 'infectious_indicators', COUNT(*),
       (SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = 'pandemic' AND table_name = 'infectious_indicators')
FROM pandemic.infectious_indicators
ORDER BY table_name;
"""
table_summary_df = query_df(TABLE_SUMMARY_SQL)
display(table_summary_df)

raw_rows_db = int(table_summary_df.loc[table_summary_df.table_name == "infectious_cases", "row_count"].iloc[0])
entity_rows_db = int(table_summary_df.loc[table_summary_df.table_name == "entities", "row_count"].iloc[0])
indicator_rows_db = int(table_summary_df.loc[table_summary_df.table_name == "infectious_indicators", "row_count"].iloc[0])

assert raw_rows_db == EXPECTED_RAW_ROWS
assert entity_rows_db == EXPECTED_ENTITY_PAIRS
assert indicator_rows_db == EXPECTED_RAW_ROWS

DUPLICATE_KEY_CHECK_SQL = r"""
SELECT COUNT(*) AS duplicate_entity_year_groups
FROM (
    SELECT entity_id, year
    FROM pandemic.infectious_indicators
    GROUP BY entity_id, year
    HAVING COUNT(*) > 1
) AS duplicates;
"""
duplicate_key_df = query_df(DUPLICATE_KEY_CHECK_SQL)
display(duplicate_key_df)
assert int(duplicate_key_df.loc[0, "duplicate_entity_year_groups"]) == 0

save_query_screenshot(
    "02 — Normalized tables in schema pandemic",
    TABLE_SUMMARY_SQL,
    table_summary_df,
    "02_normalized_tables.png",
)
table_summary_df.to_csv(RESULT_DIR / "normalized_tables.csv", index=False)

[START] Normalize raw data into 3NF tables
[DONE] Normalize raw data into 3NF tables: 0.20 sec


,table_name,row_count,column_count
0,entities,245,3
1,infectious_cases,10521,12
2,infectious_indicators,10521,11


,duplicate_entity_year_groups
0,0


Screenshot saved: /content/screenshots/02_normalized_tables.png


## 3. Статистика `Number_rabies`

Групуємо дані за `entity_id`, `entity` і `code`. Використовуємо `LEFT JOIN`, щоб запит логічно охоплював кожну комбінацію з довідника; агрегатні функції автоматично ігнорують SQL `NULL`. Для результату залишаємо 10 груп із найбільшим середнім значенням.

In [6]:
RABIES_TOP10_SQL = r"""
SELECT
    e.entity_id,
    e.entity,
    e.code,
    ROUND(AVG(i.number_rabies), 2) AS avg_number_rabies,
    MIN(i.number_rabies) AS min_number_rabies,
    MAX(i.number_rabies) AS max_number_rabies,
    SUM(i.number_rabies) AS sum_number_rabies
FROM pandemic.entities AS e
LEFT JOIN pandemic.infectious_indicators AS i
       ON i.entity_id = e.entity_id
GROUP BY e.entity_id, e.entity, e.code
ORDER BY AVG(i.number_rabies) DESC NULLS LAST, e.entity, e.code
LIMIT 10;
"""
rabies_top10_df = query_df(RABIES_TOP10_SQL)
display(rabies_top10_df)
assert len(rabies_top10_df) == 10

save_query_screenshot(
    "03 — Number_rabies: top 10 Entity + Code by average",
    RABIES_TOP10_SQL,
    rabies_top10_df,
    "03_rabies_statistics_top10.png",
)
rabies_top10_df.to_csv(RESULT_DIR / "rabies_statistics_top10.csv", index=False)

,entity_id,entity,code,avg_number_rabies,min_number_rabies,max_number_rabies,sum_number_rabies
0,102,World,OWID_WRL,20192.37,14075.508,24744.658,605771.111
1,196,Lower Middle Income (WB),None,15193.96,10202.528,19182.795,455818.7815
2,162,South Asia (WB),None,11729.89,7271.2754,15361.878,351896.6626
3,66,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.958,342729.8068
4,51,G20,None,10189.05,6339.0796,13164.881,305671.3801
5,182,India,IND,8599.17,5425.8726,11121.142,257975.1984
6,235,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.977,142448.0427
7,82,African Region (WHO),None,4649.70,4283.073,4917.784,139491.0713
8,65,Low Income (WB),None,3568.11,3014.617,3873.533,107043.1635
9,11,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


Screenshot saved: /content/screenshots/03_rabies_statistics_top10.png


## 4. Різниця в роках за допомогою вбудованих SQL-функцій

`MAKE_DATE(year, 1, 1)` створює дату 1 січня відповідного року. `CURRENT_DATE` повертає поточну дату. `AGE(current_date, year_start_date)` формує календарний інтервал, а `EXTRACT(YEAR FROM ...)` повертає кількість повних років. Зберігаємо результат у view, щоб атрибут був побудований для всіх спостережень, а не лише для прикладу на екрані.

In [7]:
YEAR_DIFFERENCE_VIEW_SQL = r"""
CREATE OR REPLACE VIEW pandemic.v_infectious_cases_year_difference AS
SELECT
    e.entity_id,
    e.entity,
    e.code,
    i.year,
    MAKE_DATE(i.year, 1, 1) AS year_start_date,
    CURRENT_DATE AS current_date,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(i.year, 1, 1)))::INTEGER
        AS year_difference
FROM pandemic.infectious_indicators AS i
JOIN pandemic.entities AS e USING (entity_id);
"""
timed_step("Create year-difference view", lambda: execute_script(YEAR_DIFFERENCE_VIEW_SQL))

YEAR_DIFFERENCE_QUERY = r"""
SELECT entity, code, year, year_start_date, current_date, year_difference
FROM pandemic.v_infectious_cases_year_difference
ORDER BY entity, year
LIMIT 20;
"""
year_difference_df = query_df(YEAR_DIFFERENCE_QUERY)
display(year_difference_df)

save_query_screenshot(
    "04 — Built-in SQL date functions and year difference",
    YEAR_DIFFERENCE_QUERY,
    year_difference_df,
    "04_year_difference_builtin.png",
)
year_difference_df.to_csv(RESULT_DIR / "year_difference_builtin.csv", index=False)

[START] Create year-difference view
[DONE] Create year-difference view: 0.01 sec


,entity,code,year,year_start_date,current_date,year_difference
0,Afghanistan,AFG,1960,1960-01-01,2026-11-07,66
1,Afghanistan,AFG,1965,1965-01-01,2026-11-07,61
2,Afghanistan,AFG,1980,1980-01-01,2026-11-07,46
3,Afghanistan,AFG,1981,1981-01-01,2026-11-07,45
4,Afghanistan,AFG,1982,1982-01-01,2026-11-07,44
5,Afghanistan,AFG,1983,1983-01-01,2026-11-07,43
6,Afghanistan,AFG,1984,1984-01-01,2026-11-07,42
7,Afghanistan,AFG,1985,1985-01-01,2026-11-07,41
8,Afghanistan,AFG,1986,1986-01-01,2026-11-07,40
9,Afghanistan,AFG,1987,1987-01-01,2026-11-07,39


Screenshot saved: /content/screenshots/04_year_difference_builtin.png


## 5. Дві власні SQL-функції

Перша функція приймає рік і повертає кількість повних років між 1 січня цього року та поточною датою. Друга функція приймає річну кількість випадків і дільник; вона повертає середню кількість випадків на період. `NULL` у будь-якому аргументі дає `NULL`, а нульовий або від’ємний дільник викликає зрозумілу помилку замість некоректного результату.

Перевіряємо функцію на прикладі `1996`, а другу функцію — для дільників `12`, `4` і `2`. Після цього застосовуємо обидві функції до реальних записів, у яких `number_rabies IS NOT NULL`.

In [8]:
FUNCTIONS_SQL = r"""
CREATE OR REPLACE FUNCTION pandemic.year_difference(p_year INTEGER)
RETURNS INTEGER
LANGUAGE SQL
STABLE
AS $function$
    SELECT EXTRACT(
        YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1))
    )::INTEGER;
$function$;

CREATE OR REPLACE FUNCTION pandemic.average_cases_per_period(
    p_annual_cases NUMERIC,
    p_divisor NUMERIC
)
RETURNS NUMERIC
LANGUAGE plpgsql
IMMUTABLE
AS $function$
BEGIN
    IF p_annual_cases IS NULL OR p_divisor IS NULL THEN
        RETURN NULL;
    END IF;

    IF p_divisor <= 0 THEN
        RAISE EXCEPTION 'The divisor must be greater than zero; received %', p_divisor
            USING ERRCODE = '22023';
    END IF;

    RETURN p_annual_cases / p_divisor;
END;
$function$;
"""
timed_step("Create custom SQL functions", lambda: execute_script(FUNCTIONS_SQL))

YEAR_FUNCTION_TEST_SQL = r"""
SELECT
    1996 AS input_year,
    MAKE_DATE(1996, 1, 1) AS year_start_date,
    CURRENT_DATE AS current_date,
    pandemic.year_difference(1996) AS year_difference;
"""
year_function_test_df = query_df(YEAR_FUNCTION_TEST_SQL)
display(year_function_test_df)

PERIOD_FUNCTION_TEST_SQL = r"""
SELECT
    120::NUMERIC AS annual_cases,
    pandemic.average_cases_per_period(120, 12) AS average_per_month,
    pandemic.average_cases_per_period(120, 4) AS average_per_quarter,
    pandemic.average_cases_per_period(120, 2) AS average_per_half_year;
"""
period_function_test_df = query_df(PERIOD_FUNCTION_TEST_SQL)
display(period_function_test_df)

assert float(period_function_test_df.loc[0, "average_per_month"]) == 10.0
assert float(period_function_test_df.loc[0, "average_per_quarter"]) == 30.0
assert float(period_function_test_df.loc[0, "average_per_half_year"]) == 60.0

CUSTOM_FUNCTIONS_QUERY = r"""
SELECT
    e.entity,
    e.code,
    i.year,
    pandemic.year_difference(i.year) AS years_since_january_first,
    i.number_rabies AS annual_rabies_cases,
    pandemic.average_cases_per_period(i.number_rabies, 12) AS average_rabies_per_month,
    pandemic.average_cases_per_period(i.number_rabies, 4) AS average_rabies_per_quarter,
    pandemic.average_cases_per_period(i.number_rabies, 2) AS average_rabies_per_half_year
FROM pandemic.infectious_indicators AS i
JOIN pandemic.entities AS e USING (entity_id)
WHERE i.number_rabies IS NOT NULL
ORDER BY e.entity, i.year
LIMIT 20;
"""
custom_functions_df = query_df(CUSTOM_FUNCTIONS_QUERY)
display(custom_functions_df)
assert len(custom_functions_df) > 0

save_query_screenshot(
    "05 — Custom SQL functions applied to numeric rabies records",
    CUSTOM_FUNCTIONS_QUERY,
    custom_functions_df,
    "05_custom_functions.png",
)
custom_functions_df.to_csv(RESULT_DIR / "custom_functions_results.csv", index=False)
year_function_test_df.to_csv(RESULT_DIR / "year_function_test.csv", index=False)
period_function_test_df.to_csv(RESULT_DIR / "period_function_test.csv", index=False)

[START] Create custom SQL functions
[DONE] Create custom SQL functions: 0.01 sec


,input_year,year_start_date,current_date,year_difference
0,1996,1996-01-01,2026-11-07,30


,annual_cases,average_per_month,average_per_quarter,average_per_half_year
0,120,10.0000000000000000,30.0000000000000000,60.0000000000000000


,entity,code,year,years_since_january_first,annual_rabies_cases,average_rabies_per_month,average_rabies_per_quarter,average_rabies_per_half_year
0,Afghanistan,AFG,1990,36,2.145772,0.17881433333333333333,0.53644300000000000000,1.07288600000000000000
1,Afghanistan,AFG,1991,35,2.3209748,0.19341456666666666667,0.58024370000000000000,1.16048740000000000000
2,Afghanistan,AFG,1992,34,2.6180408,0.21817006666666666667,0.65451020000000000000,1.30902040000000000000
3,Afghanistan,AFG,1993,33,2.8531172,0.23775976666666666667,0.71327930000000000000,1.42655860000000000000
4,Afghanistan,AFG,1994,32,2.9899871,0.24916559166666666667,0.74749677500000000000,1.49499355000000000000
5,Afghanistan,AFG,1995,31,3.126361,0.26053008333333333333,0.78159025000000000000,1.5631805000000000
6,Afghanistan,AFG,1996,30,3.281437,0.27345308333333333333,0.82035925000000000000,1.6407185000000000
7,Afghanistan,AFG,1997,29,3.446122,0.28717683333333333333,0.86153050000000000000,1.7230610000000000
8,Afghanistan,AFG,1998,28,3.5570247,0.29641872500000000000,0.88925617500000000000,1.7785123500000000
9,Afghanistan,AFG,1999,27,3.607037,0.30058641666666666667,0.90175925000000000000,1.8035185000000000


Screenshot saved: /content/screenshots/05_custom_functions.png


## 6. Підсумкові перевірки та файли доказів

Порівнюємо результат вбудованого обчислення з власною функцією для кожного рядка. Також звіряємо кількість числових значень `number_rabies` з вихідним CSV. Якщо всі `assert` проходять, імпорт, нормалізація та обидві функції працюють узгоджено.

Notebook зберігає результати SQL у `results/`, скриншоти з текстом запиту та фактичним результатом — у `screenshots/`, а потім пакує обидві папки в `goit-rdb-fp-evidence.zip`. Ці PNG можна додати до папки `screenshots/` GitHub-репозиторію.

In [9]:
FINAL_CHECK_SQL = r"""
SELECT
    (SELECT COUNT(*) FROM pandemic.infectious_cases) AS raw_rows,
    (SELECT COUNT(*) FROM pandemic.entities) AS entity_pairs,
    (SELECT COUNT(*) FROM pandemic.infectious_indicators) AS normalized_rows,
    (SELECT COUNT(*) FROM pandemic.infectious_indicators
     WHERE number_rabies IS NOT NULL) AS numeric_rabies_rows,
    (SELECT COUNT(*)
     FROM pandemic.v_infectious_cases_year_difference v
     WHERE v.year_difference IS DISTINCT FROM pandemic.year_difference(v.year)
    ) AS year_function_mismatches;
"""
final_check_df = query_df(FINAL_CHECK_SQL)
display(final_check_df)

assert int(final_check_df.loc[0, "raw_rows"]) == EXPECTED_RAW_ROWS
assert int(final_check_df.loc[0, "entity_pairs"]) == EXPECTED_ENTITY_PAIRS
assert int(final_check_df.loc[0, "normalized_rows"]) == EXPECTED_RAW_ROWS
assert int(final_check_df.loc[0, "numeric_rabies_rows"]) == EXPECTED_RABIES_VALUES
assert int(final_check_df.loc[0, "year_function_mismatches"]) == 0

final_check_df.to_csv(RESULT_DIR / "final_validation.csv", index=False)

# Bundle screenshots and query-result CSV files for convenient download from Colab.
evidence_zip = Path.cwd() / "goit-rdb-fp-evidence.zip"
with zipfile.ZipFile(evidence_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (SCREENSHOT_DIR, RESULT_DIR):
        for path in sorted(folder.glob("*")):
            if path.is_file():
                archive.write(path, arcname=f"{folder.name}/{path.name}")

print("\nУсі перевірки пройшли успішно.")
print(f"Screenshots: {SCREENSHOT_DIR.resolve()}")
print(f"SQL result CSV files: {RESULT_DIR.resolve()}")
print(f"Evidence archive: {evidence_zip.resolve()}")
print("Перед здачею додайте PNG зі screenshots/ до GitHub-репозиторію.")

,raw_rows,entity_pairs,normalized_rows,numeric_rabies_rows,year_function_mismatches
0,10521,245,10521,6780,0



Усі перевірки пройшли успішно.
Screenshots: /content/screenshots
SQL result CSV files: /content/results
Evidence archive: /content/goit-rdb-fp-evidence.zip
Перед здачею додайте PNG зі screenshots/ до GitHub-репозиторію.


## Висновок

У фінальному проєкті створено схему `pandemic`, імпортовано початкову таблицю та винесено повторювані `Entity` і `Code` у довідник. Таблиця показників посилається на довідник за `entity_id`, а пара `(entity_id, year)` однозначно ідентифікує річне спостереження. Агрегація показує розподіл `Number_rabies` за сутностями; вбудовані та власні функції повертають узгоджену кількість повних років, а розрахунок на місяць/квартал/півріччя виконується діленням річного значення на `12`, `4` і `2` відповідно.